## Загрузка окружения

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/rag_chatbot
!mkdir -p /content/drive/MyDrive/rag_chatbot/chroma_db

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#!pip uninstall -y langchain langchain-community langchain-core langchain-text-splitters langchain-openai chromadb -q

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters chromadb pypdf2 pdfplumber sentence-transformers gradio openai tiktoken requests
!pip install -q pypdf

Загрузка файла-документации

In [ ]:
import os
import requests

drive_pdf_path = "/content/drive/MyDrive/rag_chatbot/python_guide.pdf"
github_raw_url = "https://raw.githubusercontent.com/EseaHub/ML/main/RAG_ChatBot_python/Python.%20%D0%98%D1%81%D1%87%D0%B5%D1%80%D0%BF%D1%8B%D0%B2%D0%B0%D1%8E%D1%89%D0%B5%D0%B5%20%D1%80%D1%83%D0%BA%D0%BE%D0%B2%D0%BE%D0%B4%D1%81%D1%82%D0%B2%D0%BE.pdf"

In [ ]:
print("\n📚 Загружаем PDF файл...")

if os.path.exists(drive_pdf_path):
    print(" PDF уже есть в Google Drive, используем его")
    pdf_path = drive_pdf_path
else:
    print("Скачиваем PDF из GitHub...")
    try:
        response = requests.get(github_raw_url)
        response.raise_for_status()

        with open(drive_pdf_path, 'wb') as f:
            f.write(response.content)

        pdf_path = drive_pdf_path
        print(f"✅ PDF скачан и сохранен в Drive")

    except Exception as e:
        print(f"Ошибка при скачивании: {e}")
        print("Проверьте ссылку на RAW файл")
        pdf_path = None



📚 Загружаем PDF файл...
 PDF уже есть в Google Drive, используем его


Проверка читаемости файла

In [ ]:
if pdf_path and os.path.exists(pdf_path):
    import PyPDF2

    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            num_pages = len(pdf_reader.pages)
            print(f"\nКоличество страниц: {num_pages}")

            sample_text = pdf_reader.pages[55].extract_text()
            if sample_text:
                print("✅ Текст успешно извлекается")
                print(f"\nПример (первые 300 символов):")
                print(sample_text[:300] + "...")
            else:
                print("Внимание: текст может не извлекаться. Возможно, PDF отсканированный.")

    except Exception as e:
        print(f"❌ Ошибка при чтении PDF: {e}")


Количество страниц: 368
✅ Текст успешно извлекается

Пример (первые 300 символов):
56  ГлаВа 1  Основы﻿Python
1.22. PYTHON ПОДСТРАИВАЕТСЯ 
ПОД ВАШИ ЗАПРОСЫ
На заре существования Python часто приходилось слышать девиз «Он под -
страивается под ваши запросы». Даже сегодня Python — маленький язык 
программирования с полезной подборкой встроенных объектов: списков, 
множеств и словаре...


## Создаем векторную базу

In [ ]:
import os
import warnings
import requests
import pickle
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
PDF_PATH = pdf_path
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Создано чанков: {len(chunks)}")

Создано чанков: 1396


In [ ]:
print(chunks[1100].page_content[:400] + "...")

'xmlcharrefreplace'﻿
﻿
4660;'кодирование)
'surrogateescape'﻿\xhh'  
бай-
﻿\xhh'кодировании...


Создадим функцию очистки чанков

In [ ]:
import re

def clean_text(text):
  # удаляем мусорные символы
    text = re.sub(r'\\u[0-9a-fA-F]{4}', '', text)
    text = re.sub(r'&#[0-9]+;', '', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    text = re.sub(r'\s+', ' ', text)

    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        normal_chars = len(re.findall(r'[а-яА-Яa-zA-Z0-9\.,!?;:()\- ]', line))
        if len(line) > 0 and normal_chars / len(line) > 0.6:
            cleaned_lines.append(line)

    text = ' '.join(cleaned_lines)

    text = ' '.join(text.split())

    return text.strip()

In [ ]:
for chunk in chunks:
    chunk.page_content = clean_text(chunk.page_content)

chunks = [chunk for chunk in chunks if len(chunk.page_content) > 50]

print(f"После очистки осталось чанков: {len(chunks)}")

После очистки осталось чанков: 1389


In [ ]:
print(chunks[312].page_content[:400] + "...")

классы и импорт модулей, — это команды, которые по своему статусу не от- личаются от других. Это значит, что любая команда может находиться почти в любом месте программы (хотя некоторые, например return, могут быть только внутри функций). Следующий код определяет две разные версии функции внутри условной команды: if debug: def square(x): if not isinstance(x,float): raise TypeError('Expected a floa...


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
persist_directory = "/content/drive/MyDrive/rag_chatbot/chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="python_guide"
)

vectorstore.persist()

In [ ]:
test_query = "Арифметические операторы в Python"
results = vectorstore.similarity_search(test_query, k=3)

print(f"🔍 Запрос: '{test_query}'")
print(f"Найдено {len(results)} чанков")

print("\nСамый релевантный ответ:")
print(results[0].page_content[:400] + "...")


🔍 Запрос: 'Арифметические операторы в Python'
Найдено 3 чанков

Самый релевантный ответ:
Т еперь вывод программы выглядит так: 1 1050.00 2 1102.50 3 1157.62 4 1215.51 5 1276.28 1.4. АРИФМЕТИЧЕСКИЕ ОПЕРАТОРЫ Python поддерживает стандартный набор математических операторов (табл. 1.1). Их значение такое же, как и в большинстве языков программи- рования. Таблица 1.1.операторы ОПЕРаТОР ОПИСаНИЕ x + y Сложение x – y Вычитание x * y Умножение x / y Деление x // y﻿ x ** y(﻿x﻿y) x % y﻿x﻿y) –xм...


In [ ]:
import pickle

db_info = {
    'num_chunks': len(chunks),
    'num_pages': len(documents),
    'chunk_size': 500,
    'chunk_overlap': 50,
    'cleaned': True
}

with open('/content/drive/MyDrive/rag_chatbot/db_info.pkl', 'wb') as f:
    pickle.dump(db_info, f)

### Загрузка базы после перезапуска

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import torch
import pickle

device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="/content/drive/MyDrive/rag_chatbot/chroma_db",
    embedding_function=embeddings
)

with open('/content/drive/MyDrive/rag_chatbot/db_info.pkl', 'rb') as f:
    db_info = pickle.load(f)
print(f"Загружена база: {db_info['num_chunks']} чанков, {db_info['num_pages']} страниц")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_2147/4284594464.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_2147/4284594464.py:17: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Загружена база: 1389 чанков, 368 страниц
